In [1]:
# Diffusion model dependencies (TabDDPM + ForestDiffusion)
# TabDDPM: yandex-research/tab-ddpm (_vendor/tab-ddpm)
# ForestDiffusion: pip install ForestDiffusion
# libzero/rtdl pin torch<2; use --no-deps on torch 2.x (TabDDPM still works)
%pip install -q ForestDiffusion xgboost category-encoders imbalanced-learn absl-py tensorboardX icecream dython optuna skorch pyarrow tomli tomli-w
%pip install -q "pynvml>=11,<12"
%pip install -q "libzero==0.0.8" "rtdl==0.0.13" --no-deps

import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[2]
DIFFUSION_PKG = NOTEBOOK_DIR.parent
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm"))
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm" / "scripts"))
sys.path.insert(0, str(DIFFUSION_PKG))

from diffusion_generators import train_tabddpm, train_forestdiffusion

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mostlyai 5.2.5 requires requests>=2.31.0, but you have requests 2.30.0 which is incompatible.
sklearn-compat 0.1.3 requires scikit-learn<1.7,>=1.2, but you have scikit-learn 1.7.2 which is incompatible.
tensorflow-intel 2.12.0 requires numpy<1.24,>=1.22, but you have numpy 2.2.6 which is incompatible.
ydata-synthetic 1.4.0 requires numpy<2, but you have numpy 2.2.6 which is incompatible.
ydata-synthetic 1.4.0 requires tensorflow==2.15.*, but you have tensorflow 2.12.0 which is incompatible.


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
from ucimlrepo import fetch_ucirepo


In [3]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
from ucimlrepo import fetch_ucirepo
import numpy as np
import pandas as pd
from sdv.metadata import SingleTableMetadata
import torch
import random

energy_efficiency = fetch_ucirepo(id=242)
X = energy_efficiency.data.features
y = energy_efficiency.data.targets
print(energy_efficiency.metadata)
print(energy_efficiency.variables)

data = pd.concat([X, y[["Y1"]]], axis=1)
target_col = "Y1"

# Drop date/time/session/ID features before generator training (high cardinality).
_drop_feature_cols = [
    "Date", "Time", "date_time",
    "session_id", "Session ID", "Session_ID", "session",
    "X1 transaction date",
]
data = data.drop(columns=[c for c in _drop_feature_cols if c in data.columns], errors="ignore")

n_samples = min(1000, len(data))
data = data.sample(n=n_samples, random_state=42).reset_index(drop=True)

for col in data.columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")
    data[col] = data[col].fillna(data[col].median())

processed_data = data.copy()

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(processed_data)

N_SAMPLES = 1000
TEST_SIZE = 0.2
SEED = 42

# Experiment settings
FAST_MODE = True
RUN_QUALITY_EVAL = True
EVAL_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

ALL_GENERATORS = [
    'CTGAN', 'CopulaGAN', 'TVAE', 'GaussianCopula', 'TabDDPM', 'ForestDiffusion'
]
GENERATORS_TO_EVAL = ALL_GENERATORS

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

scores = {}
synthetic_datasets = {}
quality_results = []

def align_to_train_schema(df, reference_df, label_col):
    """Map synthetic data to match the schema of the reference training data."""
    df = df.copy()
    y = pd.to_numeric(df[label_col], errors='coerce').fillna(0)
    X = df.drop(columns=[label_col], errors='ignore')
    X_ref = reference_df.drop(columns=[label_col], errors='ignore')

    if X.select_dtypes(include=['object', 'string', 'category']).shape[1] > 0:
        X = pd.get_dummies(X, drop_first=True)

    X = X.reindex(columns=X_ref.columns, fill_value=0)
    X = X.apply(pd.to_numeric, errors='coerce').fillna(0).astype(np.float64)

    out = pd.concat([X.reset_index(drop=True), y.reset_index(drop=True)], axis=1)
    out.columns = reference_df.columns
    return out


{'uci_id': 242, 'name': 'Energy Efficiency', 'repository_url': 'https://archive.ics.uci.edu/dataset/242/energy+efficiency', 'data_url': 'https://archive.ics.uci.edu/static/public/242/data.csv', 'abstract': 'This study looked into assessing the heating load and cooling load requirements of buildings (that is, energy efficiency) as a function of building parameters.', 'area': 'Computer Science', 'tasks': ['Classification', 'Regression'], 'characteristics': ['Multivariate'], 'num_instances': 768, 'num_features': 8, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': ['Y1', 'Y2'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2012, 'last_updated': 'Mon Feb 26 2024', 'dataset_doi': '10.24432/C51307', 'creators': ['Athanasios Tsanas', 'Angeliki Xifara'], 'intro_paper': {'ID': 379, 'type': 'NATIVE', 'title': 'Accurate quantitative estimation of energy performance of residential buildings using statistical machine 

In [4]:
from sklearn.model_selection import train_test_split
from sdv.evaluation.single_table import evaluate_quality

seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

train_real, test_real = train_test_split(
    processed_data,
    test_size=TEST_SIZE,
        random_state=seed
)

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

if 'TabDDPM' in GENERATORS_TO_EVAL:
    import traceback
    try:
        print('Training TabDDPM...')
        synthetic_tabddpm = train_tabddpm(
            train_real,
            target_col=target_col,
            categorical_columns=[],
            n_samples=N_SAMPLES,
            seed=seed,
        )
        synthetic_datasets['TabDDPM'] = synthetic_tabddpm.copy()
        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_real,
                synthetic_data=synthetic_tabddpm,
                metadata=train_metadata,
            )
            scores['TabDDPM'] = quality.get_score()
            print('TabDDPM:', round(scores['TabDDPM'], 4))
        else:
            print('TabDDPM: trained (quality eval skipped)')
    except Exception as e:
        print('TabDDPM Failed:', e)
        traceback.print_exc()
else:
    print('TabDDPM: skipped (not in GENERATORS_TO_EVAL)')


================ SINGLE RUN ================
Training TabDDPM...
[0]
9
{'num_classes': 0, 'is_y_cond': False, 'rtdl_params': {'d_layers': [256, 256, 256], 'dropout': 0.0}, 'd_in': np.int64(9)}
mlp
Step 500/1000 MLoss: 0.0 GLoss: 0.4302 Sum: 0.4302
Step 1000/1000 MLoss: 0.0 GLoss: 0.3453 Sum: 0.3453
mlp
Sample timestep    0
Discrete cols: [5, 7]
Num shape:  (1000, 8)
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 269.00it/s]|
Column Shapes Score: 70.49%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 184.76it/s]|
Column Pair Trends Score: 73.01%

Overall Score (Average): 71.75%

TabDDPM: 0.7175


In [5]:
# ForestDiffusion
if 'ForestDiffusion' in GENERATORS_TO_EVAL:
    import traceback
    try:
        print('Training ForestDiffusion...')
        synthetic_forestdiffusion = train_forestdiffusion(
            train_real,
            target_col=target_col,
            categorical_columns=[target_col],
            n_samples=N_SAMPLES,
            seed=seed,
        )
        synthetic_datasets['ForestDiffusion'] = synthetic_forestdiffusion.copy()
        print('ForestDiffusion: synthesis complete')
        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_real,
                synthetic_data=synthetic_forestdiffusion,
                metadata=train_metadata,
            )
            scores['ForestDiffusion'] = quality.get_score()
            print('ForestDiffusion:', round(scores['ForestDiffusion'], 4))
        else:
            print('ForestDiffusion: trained (quality eval skipped)')
    except Exception as e:
        print('ForestDiffusion Failed (training/sampling):')
        traceback.print_exc()
    if 'ForestDiffusion' in synthetic_datasets and RUN_QUALITY_EVAL:
        pass
else:
    print('ForestDiffusion: skipped (not in GENERATORS_TO_EVAL)')

Training ForestDiffusion...
ForestDiffusion: synthesis complete
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 121.40it/s]|
Column Shapes Score: 67.56%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 156.64it/s]|
Column Pair Trends Score: 60.2%

Overall Score (Average): 63.88%

ForestDiffusion: 0.6388


In [6]:
# SDV MODELS

from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer
)

from sdv.evaluation.single_table import evaluate_quality

sdv_models = {
    "CTGAN": CTGANSynthesizer(metadata=train_metadata),
    "CopulaGAN": CopulaGANSynthesizer(metadata=train_metadata),
    "TVAE": TVAESynthesizer(metadata=train_metadata),
    "GaussianCopula": GaussianCopulaSynthesizer(metadata=train_metadata)
}

for model_name, model in sdv_models.items():

    try:

        model.fit(train_real)

        synthetic_data = model.sample(N_SAMPLES)

        pass  # keep continuous regression target as-is

        synthetic_datasets[model_name] = synthetic_data.copy()

        quality = evaluate_quality(
            real_data=train_real,
            synthetic_data=synthetic_data,
            metadata=train_metadata
        )

        scores[model_name] = quality.get_score()

        print(
            f"{model_name}: {round(scores[model_name], 4)}"
        )

    except Exception as e:

        print(
            f"{model_name} Failed: {e}"
        )

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 241.33it/s]|
Column Shapes Score: 82.06%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 285.24it/s]|
Column Pair Trends Score: 70.61%

Overall Score (Average): 76.33%

CTGAN: 0.7633
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 289.45it/s]|
Column Shapes Score: 75.85%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 319.82it/s]|
Column Pair Trends Score: 68.73%

Overall Score (Average): 72.29%

CopulaGAN: 0.7229
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 968.11it/s]|
Column Shapes Score: 84.17%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 270.14it/s]|
Column Pair Trends Score: 78.99%

Overall Score (Average): 81.58%

TVAE: 0.8158
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 270.08it/s]|
Column Shapes Score: 7

In [7]:
from sklearn.base import clone
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor

regressors = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.001, max_iter=5000),
    'ElasticNet': ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000),
    'SVR_RBF': SVR(kernel='rbf', C=1.0, epsilon=0.1),
    'KNN': KNeighborsRegressor(n_neighbors=5),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'GradientBoost': GradientBoostingRegressor(random_state=42),
}

print(f'Regression evaluation: {len(regressors)} models, {len(EVAL_SEEDS)} seeds, {len(GENERATORS_TO_EVAL)} generators')


Regression evaluation: 10 models, 10 seeds, 6 generators


In [14]:
def evaluate_regression_models(train_df, test_df, label_col, models, test_size=0.2, seeds=None, use_holdout=False, schema_df=None):
    if seeds is None:
        seeds = EVAL_SEEDS
    if schema_df is None:
        schema_df = test_df if use_holdout else train_df

    train_df = align_to_train_schema(train_df, schema_df, label_col)
    test_df = align_to_train_schema(test_df, schema_df, label_col)
    results = []

    for name, model in models.items():
        r2_scores = []
        mse_scores = []
        rmse_scores = []
        mae_scores = []

        for seed in seeds:
            X_train = train_df.drop(columns=[label_col])
            y_train = train_df[label_col]

            X_test = test_df.drop(columns=[label_col])
            y_test = test_df[label_col]

            if use_holdout:
                # Keep holdout protocol, but bootstrap train/test rows per seed
                # so repeated runs produce meaningful variance estimates.
                rng = np.random.default_rng(seed)
                train_idx = rng.choice(len(X_train), size=len(X_train), replace=True)
                test_idx = rng.choice(len(X_test), size=len(X_test), replace=True)

                X_train = X_train.iloc[train_idx].reset_index(drop=True)
                y_train = y_train.iloc[train_idx].reset_index(drop=True)
                X_test = X_test.iloc[test_idx].reset_index(drop=True)
                y_test = y_test.iloc[test_idx].reset_index(drop=True)
            else:
                X_train, _, y_train, _ = train_test_split(
                    X_train, y_train, test_size=test_size, random_state=seed
                )
                _, X_test, _, y_test = train_test_split(
                    X_test, y_test, test_size=test_size, random_state=seed
                )

            reg = clone(model)
            if 'random_state' in reg.get_params():
                reg.set_params(random_state=seed)

            reg.fit(X_train, y_train)
            y_pred = reg.predict(X_test)

            mse = mean_squared_error(y_test, y_pred)
            r2_scores.append(r2_score(y_test, y_pred))
            mse_scores.append(mse)
            rmse_scores.append(np.sqrt(mse))
            mae_scores.append(mean_absolute_error(y_test, y_pred))

        results.append({
            'Model': name,
            'R2 Mean': np.mean(r2_scores),
            'R2 Std': np.std(r2_scores),
            'MSE Mean': np.mean(mse_scores),
            'MSE Std': np.std(mse_scores),
            'RMSE Mean': np.mean(rmse_scores),
            'RMSE Std': np.std(rmse_scores),
            'MAE Mean': np.mean(mae_scores),
            'MAE Std': np.std(mae_scores),
            'R2 (Mean±Std)': f"{np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}",
            'MSE (Mean±Std)': f"{np.mean(mse_scores):.4f} ± {np.std(mse_scores):.4f}",
            'RMSE (Mean±Std)': f"{np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}",
            'MAE (Mean±Std)': f"{np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}",
        })

    return pd.DataFrame(results).sort_values(by='R2 Mean', ascending=False)


In [15]:
print('TRTR (Train Real, Test Real) — 80% train / 20% holdout')
trtr_results = evaluate_regression_models(
    train_df=train_real,
    test_df=test_real,
    label_col=target_col,
    models=regressors,
    seeds=EVAL_SEEDS,
    use_holdout=True,
    schema_df=train_real,
)
display(trtr_results[['Model', 'R2 (Mean±Std)', 'MSE (Mean±Std)', 'RMSE (Mean±Std)', 'MAE (Mean±Std)']])

all_comparisons = []
for synth_name in GENERATORS_TO_EVAL:
    if synth_name not in synthetic_datasets:
        print(f'Skipping {synth_name} - no synthetic dataset')
        continue

    print(f'{synth_name} - TSTR (train on synthetic, test on 20% holdout)')
    tstr_results = evaluate_regression_models(
        train_df=synthetic_datasets[synth_name],
        test_df=test_real,
        label_col=target_col,
        models=regressors,
        seeds=EVAL_SEEDS,
        use_holdout=True,
        schema_df=train_real,
    )
    display(tstr_results[['Model', 'R2 (Mean±Std)', 'MSE (Mean±Std)', 'RMSE (Mean±Std)', 'MAE (Mean±Std)']])

    comparison = trtr_results.merge(tstr_results, on='Model', suffixes=('_TRTR', '_TSTR'))
    comparison['R2_Drop'] = comparison['R2 Mean_TRTR'] - comparison['R2 Mean_TSTR']
    comparison['MSE_Increase'] = comparison['MSE Mean_TSTR'] - comparison['MSE Mean_TRTR']
    comparison['RMSE_Increase'] = comparison['RMSE Mean_TSTR'] - comparison['RMSE Mean_TRTR']
    comparison['MAE_Increase'] = comparison['MAE Mean_TSTR'] - comparison['MAE Mean_TRTR']
    comparison['Synthetic_Model'] = synth_name
    all_comparisons.append(comparison)

combined_comparison = pd.concat(all_comparisons, ignore_index=True)
summary = (
    combined_comparison
    .groupby('Synthetic_Model', as_index=False)[['R2_Drop', 'MSE_Increase', 'RMSE_Increase', 'MAE_Increase']]
    .mean()
    .sort_values('R2_Drop')
)

display(summary)


TRTR (Train Real, Test Real) — 80% train / 20% holdout


,Model,R2 (Mean±Std),MSE (Mean±Std),RMSE (Mean±Std),MAE (Mean±Std)
9,GradientBoost,0.9964 ± 0.0008,0.3541 ± 0.0818,0.5911 ± 0.0680,0.4188 ± 0.0443
8,ExtraTrees,0.9963 ± 0.0015,0.3642 ± 0.1266,0.5960 ± 0.0945,0.4042 ± 0.0541
7,RandomForest,0.9958 ± 0.0018,0.4126 ± 0.1488,0.6340 ± 0.1035,0.4192 ± 0.0429
6,DecisionTree,0.9948 ± 0.0035,0.5002 ± 0.3142,0.6843 ± 0.1788,0.4436 ± 0.0759
5,KNN,0.9344 ± 0.0154,6.4109 ± 1.2005,2.5211 ± 0.2339,1.9046 ± 0.0901
0,LinearRegression,0.8895 ± 0.0131,10.9555 ± 1.6570,3.3001 ± 0.2541,2.3733 ± 0.2418
2,Lasso,0.8888 ± 0.0131,11.0253 ± 1.6499,3.3108 ± 0.2524,2.3864 ± 0.2431
3,ElasticNet,0.8840 ± 0.0129,11.4995 ± 1.6192,3.3824 ± 0.2424,2.5019 ± 0.2344
1,Ridge,0.8829 ± 0.0133,11.6050 ± 1.6399,3.3978 ± 0.2446,2.5370 ± 0.2359
4,SVR_RBF,0.6588 ± 0.0497,33.9222 ± 6.1562,5.7989 ± 0.5433,4.0986 ± 0.3540


CTGAN - TSTR (train on synthetic, test on 20% holdout)


,Model,R2 (Mean±Std),MSE (Mean±Std),RMSE (Mean±Std),MAE (Mean±Std)
9,GradientBoost,-0.0707 ± 0.3235,105.9116 ± 31.1332,10.1664 ± 1.5984,8.3799 ± 1.4026
7,RandomForest,-0.0937 ± 0.2951,107.7279 ± 27.3273,10.2977 ± 1.2980,8.5236 ± 1.0702
8,ExtraTrees,-0.1631 ± 0.2232,114.5287 ± 19.7291,10.6641 ± 0.8977,9.0749 ± 0.7574
1,Ridge,-0.3049 ± 0.1464,128.5856 ± 10.1219,11.3311 ± 0.4385,9.6958 ± 0.4604
3,ElasticNet,-0.3100 ± 0.1482,129.0728 ± 10.2179,11.3524 ± 0.4416,9.7201 ± 0.4626
2,Lasso,-0.3155 ± 0.1501,129.6079 ± 10.3280,11.3758 ± 0.4453,9.7466 ± 0.4652
0,LinearRegression,-0.3171 ± 0.1507,129.7619 ± 10.3753,11.3825 ± 0.4470,9.7542 ± 0.4659
4,SVR_RBF,-0.3620 ± 0.1018,134.7140 ± 10.8448,11.5970 ± 0.4719,9.1786 ± 0.4921
5,KNN,-0.8423 ± 0.2901,182.2184 ± 28.7181,13.4557 ± 1.0781,10.8308 ± 0.9372
6,DecisionTree,-1.0256 ± 0.5147,199.0285 ± 46.3038,14.0113 ± 1.6465,11.1360 ± 1.4642


CopulaGAN - TSTR (train on synthetic, test on 20% holdout)


...

TabDDPM - TSTR (train on synthetic, test on 20% holdout)


,Model,R2 (Mean±Std),MSE (Mean±Std),RMSE (Mean±Std),MAE (Mean±Std)
8,ExtraTrees,0.8999 ± 0.0262,9.8690 ± 2.5972,3.1149 ± 0.4076,2.0419 ± 0.2791
9,GradientBoost,0.8872 ± 0.0304,11.0960 ± 2.7013,3.3062 ± 0.4067,2.4039 ± 0.3445
7,RandomForest,0.8758 ± 0.0191,12.2647 ± 1.7542,3.4924 ± 0.2606,2.1661 ± 0.1824
5,KNN,0.8526 ± 0.0184,14.5670 ± 1.7099,3.8100 ± 0.2263,2.8548 ± 0.1779
6,DecisionTree,0.8306 ± 0.0438,16.6234 ± 3.5307,4.0570 ± 0.4054,2.5714 ± 0.2846
0,LinearRegression,0.5328 ± 0.0571,46.2160 ± 5.8713,6.7842 ± 0.4369,5.2910 ± 0.3220
2,Lasso,0.5316 ± 0.0573,46.3381 ± 5.8952,6.7931 ± 0.4381,5.2971 ± 0.3225
3,ElasticNet,0.5216 ± 0.0576,47.3284 ± 5.9412,6.8657 ± 0.4373,5.3401 ± 0.3185
1,Ridge,0.5175 ± 0.0578,47.7368 ± 5.9978,6.8952 ± 0.4398,5.3508 ± 0.3189
4,SVR_RBF,-0.0186 ± 0.2632,100.0930 ± 23.8471,9.9270 ± 1.2437,7.3320 ± 1.0645


ForestDiffusion - TSTR (train on synthetic, test on 20% holdout)


,Model,R2 (Mean±Std),MSE (Mean±Std),RMSE (Mean±Std),MAE (Mean±Std)
8,ExtraTrees,0.9960 ± 0.0007,0.3988 ± 0.0710,0.6290 ± 0.0560,0.4336 ± 0.0380
7,RandomForest,0.9952 ± 0.0010,0.4739 ± 0.1078,0.6843 ± 0.0750,0.4685 ± 0.0430
9,GradientBoost,0.9952 ± 0.0008,0.4771 ± 0.0767,0.6885 ± 0.0556,0.4815 ± 0.0315
6,DecisionTree,0.9939 ± 0.0022,0.6120 ± 0.2475,0.7699 ± 0.1387,0.5128 ± 0.0486
5,KNN,0.9227 ± 0.0092,7.6335 ± 0.8631,2.7588 ± 0.1506,2.1862 ± 0.0943
0,LinearRegression,0.8905 ± 0.0131,10.8792 ± 1.6609,3.2880 ± 0.2615,2.3814 ± 0.2154
2,Lasso,0.8900 ± 0.0131,10.9318 ± 1.6657,3.2960 ± 0.2615,2.3884 ± 0.2150
3,ElasticNet,0.8867 ± 0.0130,11.2594 ± 1.6706,3.3456 ± 0.2577,2.4362 ± 0.2055
1,Ridge,0.8854 ± 0.0130,11.3815 ± 1.6753,3.3639 ± 0.2568,2.4516 ± 0.2032
4,SVR_RBF,0.7187 ± 0.0266,27.9430 ± 3.6587,5.2740 ± 0.3575,3.6558 ± 0.2880


,Synthetic_Model,R2_Drop,MSE_Increase,RMSE_Increase,MAE_Increase
2,ForestDiffusion,-0.005265,-0.505924,-0.011870,-0.009157
4,TVAE,0.142111,14.082883,2.270088,1.599295
5,TabDDPM,0.269074,26.508303,3.082895,2.316170
3,GaussianCopula,0.278949,27.524174,3.544535,3.293261
1,CopulaGAN,1.279532,126.566885,9.113273,8.515516
0,CTGAN,1.292640,127.410771,9.141758,7.855306


In [16]:
# Create quality metrics DataFrame
quality_df = pd.DataFrame({
    'Generator': list(scores.keys()),
    'Quality_Score': list(scores.values())
})

output_file = 'TRTR_TSTR_results_energy_efficiency.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    quality_df.to_excel(writer, sheet_name='Quality_Metrics', index=False)
    trtr_results.to_excel(writer, sheet_name='TRTR_Results', index=False)
    combined_comparison.to_excel(writer, sheet_name='All_Comparisons', index=False)
    summary.to_excel(writer, sheet_name='Summary', index=False)
    for synth_name in GENERATORS_TO_EVAL:
        if synth_name in combined_comparison['Synthetic_Model'].values:
            synth_results = combined_comparison[combined_comparison['Synthetic_Model'] == synth_name]
            synth_results.to_excel(writer, sheet_name=synth_name[:31], index=False)

print(f'Results saved to: {output_file}')


Results saved to: TRTR_TSTR_results_energy_efficiency.xlsx
